# 01 — Export random-window data (Shalow)

Pipeline step 1: export **one random pool** of sequences from 2013–2022, then split **70/30 train/val at training time** in `04_Sh_train_magic_window.ipynb`.

**Settings**
- `DT = 360`, 3 cond + 1 pred, u+v
- **Pool:** noon-only (`HOUR_OF_DAY = 12`), **K=2 crops per anchor** → up to **~8000 rows** (`sample_13_22_uv_dt360_*`)
- **Spatial:** two independent random 128×128 crops per noon start → `{prefix}_windows.npy` (frozen val crops)
- **Per-epoch train windows:** after pool merge, pre-export random crops for **epochs 1…`N_EXPORT_EPOCHS`** for **k1 and k2** → `{prefix}_windows_per_epoch_k1.npy`, `_k2.npy` (training reads these instead of re-randomizing each epoch)
- **Mask filter:** drop if land/coastal invalid **≥ 5%** (`MAX_MASK_FRAC = 0.05`)
- **Test:** 2023 noon full year, **1 crop per anchor** → `sample_23_uv_dt360_*`

**Must match notebook 04:** `N_EXPORT_EPOCHS`, `RANDOM_CROP_SEED`, `VAL_SPLIT`, `VAL_SPLIT_MODE`, `VAL_SPLIT_SEED`.

**Output layout** — `Shalow/data/{export_key}/`:
```
sample_13_22_uv_dt360_*              # training pool (no export-time val split)
sample_13_22_uv_dt360_windows_per_epoch_k1.npy   # (n_epochs, n_train, 4)
sample_13_22_uv_dt360_windows_per_epoch_k2.npy
sample_13_22_uv_dt360_epoch_windows_meta.json
sample_13_22_export.json             # export metadata + unique zarr anchors
sample_23_uv_dt360_*                 # 2023 rollout test
```


In [1]:
import json
import os
import sys
from dataclasses import asdict
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
SHALOW_DIR = NOTEBOOK_DIR if NOTEBOOK_DIR.name == "Shalow" else NOTEBOOK_DIR / "Shalow"
ROOT = SHALOW_DIR.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SHALOW_DIR) not in sys.path:
    sys.path.insert(0, str(SHALOW_DIR))

DATA_DIR = SHALOW_DIR / "data"
LOG_DIR = SHALOW_DIR / "logs"
for _d in (DATA_DIR, LOG_DIR):
    _d.mkdir(parents=True, exist_ok=True)

from src.data.dataloaders.guiana_margem_tensor import GuianaDataloader
from src.export.fix_window_export import candidate_indices_for_hour
from src.export.random_window_export import (
    RandomWindowExportConfig,
    export_wants_fresh_start,
    plan_multi_crop_export,
    run_export,
)
from window_paths import export_manifest_path, pool_prefix, test_prefix, window_data_dir


In [2]:
ZARR_PATH = os.getenv(
    "GUIANA_ZARR_PATH",
    "s3://oceangen-data/margem_equatorial_tensor.zarr",
)

EXPORT_KEY = "random_win_128"
CROP_HEIGHT = 128
CROP_WIDTH = 128

DT = 360
CONDITIONING_STEPS = 3
FORECAST_STEPS = 1
EXPORT_COMPONENTS = ("uv",)

WINDOW_DATA_DIR = window_data_dir(DATA_DIR, EXPORT_KEY)
POOL_DATA_PREFIX_BASE = pool_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
TEST_DATA_PREFIX_BASE = test_prefix(DATA_DIR, EXPORT_KEY, dt=DT)
EXPORT_MANIFEST_PATH = export_manifest_path(DATA_DIR, EXPORT_KEY)


def _component_export_settings(component: str, base: str | None = None):
    if component == "uv":
        base = base or POOL_DATA_PREFIX_BASE
        return base, ("u", "v"), (0, 1)
    raise ValueError(f"Unknown component {component!r}; use 'uv'")


PREVIEW_COMPONENT = EXPORT_COMPONENTS[0]
OUTPUT_PREFIX, CHANNEL_NAMES, TARGET_CHANNEL_INDICES = _component_export_settings(PREVIEW_COMPONENT)

SEED = 42
N_JOBS = 30
MAX_MASK_FRAC = 0.05  # reject crop when masked fraction >= 5%; keep strictly ocean-heavy patches
MAX_CROP_ATTEMPTS = 200
HOUR_OF_DAY = 12  # noon-only anchors
CROPS_PER_ANCHOR = 2  # K independent random crops per noon start

# Per-epoch train windows (must match Shalow/04_Sh_train_magic_window.ipynb)
N_EXPORT_EPOCHS = 100
RANDOM_CROP_SEED = 42
VAL_SPLIT = 0.3
VAL_SPLIT_MODE = "random"  # "random" | "chronological"
VAL_SPLIT_SEED = 42
EXPORT_EPOCH_WINDOWS = True
TRAIN_CROPS_FOR_EPOCH_WINDOWS = (1, 2)  # k1 and k2

N_SAMPLES_TO_GEN = 8000  # cap on export rows (= n_anchors * CROPS_PER_ANCHOR)

START_TIME = "2013-01-01"
END_TIME = "2022-12-31"

WINDOW_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [3]:
base_kwargs = dict(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=CONDITIONING_STEPS,
    forecast_steps=FORECAST_STEPS,
    start_time=START_TIME,
    end_time=END_TIME,
)

print("Opening Zarr:", ZARR_PATH)
dataset = GuianaDataloader(**base_kwargs, transform=None)
info = dataset.info()
candidate_idx = candidate_indices_for_hour(dataset, HOUR_OF_DAY)

pool_anchors, pool_zarr_idx, pool_crop_slots = plan_multi_crop_export(
    candidate_idx,
    n_samples_to_gen=N_SAMPLES_TO_GEN,
    crops_per_anchor=CROPS_PER_ANCHOR,
    seed=SEED,
)
n_anchors = len(pool_anchors)
n_rows = len(pool_zarr_idx)

export_manifest = {
    "export_key": EXPORT_KEY,
    "window_data_dir": str(WINDOW_DATA_DIR),
    "window_mode": "random",
    "mode": "random_pool_export",
    "sample_seed": SEED,
    "n_pool": int(len(candidate_idx)),
    "n_anchors": int(n_anchors),
    "crops_per_anchor": int(CROPS_PER_ANCHOR),
    "n_rows": int(n_rows),
    "start_time": START_TIME,
    "end_time": END_TIME,
    "hour_of_day": HOUR_OF_DAY,
    "crop_height": CROP_HEIGHT,
    "crop_width": CROP_WIDTH,
    "max_mask_frac": MAX_MASK_FRAC,
    "pool_data_prefix_base": POOL_DATA_PREFIX_BASE,
    "train_val_split_note": "70/30 split at training time in 04_Sh_train_magic_window.ipynb",
    "n_export_epochs": N_EXPORT_EPOCHS,
    "random_crop_seed": RANDOM_CROP_SEED,
    "val_split": VAL_SPLIT,
    "val_split_mode": VAL_SPLIT_MODE,
    "val_split_seed": VAL_SPLIT_SEED,
    "export_epoch_windows": EXPORT_EPOCH_WINDOWS,
    "train_crops_for_epoch_windows": list(TRAIN_CROPS_FOR_EPOCH_WINDOWS),
    "zarr_indices": pool_anchors.tolist(),
}
EXPORT_MANIFEST_PATH.write_text(json.dumps(export_manifest, indent=2))

print("Valid samples:", len(dataset))
print("After hour filter:", len(candidate_idx), "| HOUR_OF_DAY =", HOUR_OF_DAY)
print(
    f"Pool plan: {n_anchors} noon anchors × {CROPS_PER_ANCHOR} crops = {n_rows} export rows "
    f"(seed={SEED}) → {POOL_DATA_PREFIX_BASE}"
)
print("Export manifest:", EXPORT_MANIFEST_PATH)
print("Grid (H, W):", info["space"]["height"], info["space"]["width"])
print(f"Crop: {CROP_HEIGHT}x{CROP_WIDTH} random | keep if land/invalid < {MAX_MASK_FRAC:.0%}")
if EXPORT_EPOCH_WINDOWS:
    print(
        f"Per-epoch train windows: {N_EXPORT_EPOCHS} epochs, k={TRAIN_CROPS_FOR_EPOCH_WINDOWS}, "
        f"split={VAL_SPLIT_MODE} ({int((1-VAL_SPLIT)*100)}% train), seed={RANDOM_CROP_SEED}"
    )
if n_rows < N_SAMPLES_TO_GEN:
    print(
        f"NOTE: {n_rows} rows < N_SAMPLES_TO_GEN={N_SAMPLES_TO_GEN} "
        f"(limited by {len(candidate_idx)} noon anchors × {CROPS_PER_ANCHOR})"
    )


Opening Zarr: s3://oceangen-data/margem_equatorial_tensor.zarr
Valid samples: 87648
After hour filter: 3652 | HOUR_OF_DAY = 12
Pool plan: 3652 noon anchors × 2 crops = 7304 export rows (seed=42) → /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360
Export manifest: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_export.json
Grid (H, W): 431 547
Crop: 128x128 random | keep if land/invalid < 5%
NOTE: 7304 rows < N_SAMPLES_TO_GEN=8000 (limited by 3652 noon anchors × 2)


## Parallel export (resumable)

One export job for the 2013–2022 pool. Re-run or use the CLI to resume.

After the pool merge finishes, **per-epoch train windows** (`_windows_per_epoch_k1.npy`, `_k2.npy`) are generated automatically. If the pool is already exported, run only that step:

```bash
python scripts/export_random_window_data.py --output-prefix <POOL_PREFIX> --epoch-windows-only
```


In [4]:
import subprocess

DETACHED = True
CHUNK_SIZE = 100
RESUME = True


def _run_one_export(
    output_prefix: str,
    channel_names: tuple[str, ...],
    target_channel_indices: tuple[int, ...],
    *,
    start_time: str | None = None,
    end_time: str | None = None,
    n_samples_to_gen: int | None = None,
    crops_per_anchor: int | None = None,
    fixed_dataset_indices: tuple[int, ...] | None = None,
):
    export_config = RandomWindowExportConfig(
        zarr_path=ZARR_PATH,
        dt=DT,
        conditioning_steps=CONDITIONING_STEPS,
        forecast_steps=FORECAST_STEPS,
        crop_height=CROP_HEIGHT,
        crop_width=CROP_WIDTH,
        seed=SEED,
        n_jobs=N_JOBS,
        max_mask_frac=MAX_MASK_FRAC,
        max_crop_attempts=MAX_CROP_ATTEMPTS,
        hour_of_day=HOUR_OF_DAY,
        start_time=start_time or START_TIME,
        end_time=end_time or END_TIME,
        output_prefix=output_prefix,
        n_samples_to_gen=n_samples_to_gen if n_samples_to_gen is not None else N_SAMPLES_TO_GEN,
        crops_per_anchor=crops_per_anchor if crops_per_anchor is not None else CROPS_PER_ANCHOR,
        chunk_size=CHUNK_SIZE,
        channel_names=channel_names,
        target_channel_indices=target_channel_indices,
        fixed_dataset_indices=fixed_dataset_indices,
        export_key=EXPORT_KEY,
        export_epoch_windows=EXPORT_EPOCH_WINDOWS,
        n_export_epochs=N_EXPORT_EPOCHS,
        random_crop_seed=RANDOM_CROP_SEED,
        val_split=VAL_SPLIT,
        val_split_mode=VAL_SPLIT_MODE,
        val_split_seed=VAL_SPLIT_SEED,
        train_crops_for_epoch_windows=TRAIN_CROPS_FOR_EPOCH_WINDOWS,
    )

    config_path = Path(f"{output_prefix}_export_config.json")
    config_dict = asdict(export_config)
    config_dict["window_mode"] = "random"
    config_path.write_text(json.dumps(config_dict, indent=2, default=str))
    print(f"Wrote config: {config_path}")

    fresh_start = export_wants_fresh_start(export_config, resume=RESUME)

    if DETACHED:
        log_path = LOG_DIR / f"export_{EXPORT_KEY}_{Path(output_prefix).name}.log"
        cmd = [
            sys.executable,
            str(ROOT / "scripts/export_random_window_data.py"),
            "--output-prefix",
            output_prefix,
        ]
        if fresh_start:
            cmd.append("--no-resume")
            if log_path.exists():
                log_path.unlink()
                print(f"Cleared log: {log_path}")
        log_mode = "w" if fresh_start else "a"
        with open(log_path, log_mode) as log_file:
            proc = subprocess.Popen(
                cmd,
                stdout=log_file,
                stderr=subprocess.STDOUT,
                start_new_session=True,
                cwd=ROOT,
            )
        print(f"Started background export PID {proc.pid} -> {output_prefix}")
        if fresh_start:
            print("Fresh export (log cleared, partial state reset if config changed)")
        else:
            print(f"Resuming export (appending to {log_path})")
        print(f"Log: {log_path}")
        print(f"Monitor: tail -f {log_path}")
        print(
            f"Resume: python scripts/export_random_window_data.py --output-prefix {output_prefix}"
        )
    else:
        run_export(export_config, resume=not fresh_start)


for component in EXPORT_COMPONENTS:
    pool_pfx, ch_names, tgt_idx = _component_export_settings(component, POOL_DATA_PREFIX_BASE)
    print(
        f"\n=== Pool export: {component} -> {pool_pfx} "
        f"({n_anchors} anchors × {CROPS_PER_ANCHOR} = {n_rows} rows) ==="
    )
    _run_one_export(
        pool_pfx,
        ch_names,
        tgt_idx,
        n_samples_to_gen=n_rows,
        crops_per_anchor=CROPS_PER_ANCHOR,
        fixed_dataset_indices=tuple(int(x) for x in pool_anchors),
    )



=== Pool export: uv -> /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360 (3652 anchors × 2 = 7304 rows) ===
Wrote config: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360_export_config.json
Cleared log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_sample_13_22_uv_dt360.log
Started background export PID 638648 -> /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_dt360
Fresh export (log cleared, partial state reset if config changed)
Log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_sample_13_22_uv_dt360.log
Monitor: tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_sample_13_22_uv_dt360.log
Resume: python scripts/export_random_window_data.py --output-prefix /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_13_22_uv_

## 2023 test export (rollout holdout)


In [5]:
TEST_START_TIME = "2023-01-01"
TEST_END_TIME = "2023-12-31"
TEST_EXPORT_ALL = True

test_kwargs = dict(
    path=ZARR_PATH,
    dt=DT,
    conditioning_steps=CONDITIONING_STEPS,
    forecast_steps=FORECAST_STEPS,
    start_time=TEST_START_TIME,
    end_time=TEST_END_TIME,
)
test_dataset = GuianaDataloader(**test_kwargs, transform=None)
test_candidate_idx = candidate_indices_for_hour(test_dataset, HOUR_OF_DAY)
print(f"2023 test pool: {len(test_candidate_idx)} starts | HOUR_OF_DAY={HOUR_OF_DAY}")
test_n = len(test_candidate_idx) if TEST_EXPORT_ALL else min(N_SAMPLES_TO_GEN, len(test_candidate_idx))
print(f"Will export: {test_n} test sequences")

for component in EXPORT_COMPONENTS:
    test_pfx, test_ch_names, test_tgt_idx = _component_export_settings(
        component, base=TEST_DATA_PREFIX_BASE
    )
    print(f"\n=== 2023 test export: {component} -> {test_pfx} ===")
    _run_one_export(
        test_pfx,
        test_ch_names,
        test_tgt_idx,
        start_time=TEST_START_TIME,
        end_time=TEST_END_TIME,
        n_samples_to_gen=test_n,
        crops_per_anchor=1,
        fixed_dataset_indices=tuple(int(x) for x in test_candidate_idx) if TEST_EXPORT_ALL else None,
    )


2023 test pool: 319 starts | HOUR_OF_DAY=12
Will export: 319 test sequences

=== 2023 test export: uv -> /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_23_uv_dt360 ===
Wrote config: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_23_uv_dt360_export_config.json
Cleared log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_sample_23_uv_dt360.log
Started background export PID 638794 -> /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data/random_win_128/sample_23_uv_dt360
Fresh export (log cleared, partial state reset if config changed)
Log: /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_sample_23_uv_dt360.log
Monitor: tail -f /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/logs/export_random_win_128_sample_23_uv_dt360.log
Resume: python scripts/export_random_window_data.py --output-prefix /opt/dlami/nvme/tmp_storage/LCADAME/hamid_copy/Shalow/data